<a href="https://colab.research.google.com/github/lukalklikadze/walmart-sales-forecasting/blob/main/notebooks/model_experiment_TFT.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip -q install mlflow kaggle

!git clone -q https://github.com/lukalklikadze/walmart-sales-forecasting.git /content/repo \
    2>/dev/null || (cd /content/repo && git pull -q)
import sys; sys.path.append("/content/repo")

from src.config import setup_env, DATA_DIR, KAGGLE_COMP
tracking_uri = setup_env()
print("MLflow →", tracking_uri)

import os, glob, zipfile, subprocess
os.makedirs(DATA_DIR, exist_ok=True)
subprocess.run(["kaggle","competitions","download","-c",KAGGLE_COMP,"-p",DATA_DIR,"--force"], check=True)
with zipfile.ZipFile(os.path.join(DATA_DIR, KAGGLE_COMP + ".zip")) as z: z.extractall(DATA_DIR)
for inner in glob.glob(os.path.join(DATA_DIR, "*.csv.zip")):
    with zipfile.ZipFile(inner) as z: z.extractall(DATA_DIR)

from src.data import load_raw
train, test, stores, features = load_raw()
print("train:", train.shape, "| test:", test.shape)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 2.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.5/50.5 kB 3.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.6/12.6 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 69.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 12.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 148.8/148.8 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 114.9/114.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 212.0/212.0 kB 11.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.0/121.0 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.

In [2]:
# ── TFT config ──────────────────────────────────────────────────────
import numpy as np, pandas as pd, torch, torch.nn as nn, mlflow
from torch.utils.data import Dataset, DataLoader

from src.features import CalendarFeatures, GROUP_COLS
from src.metrics import wmae
from src.train_val_split import time_based_split

ENC_LEN     = 52       # 1 year of weekly lookback (encoder length)
# Same reasoning as the DLinear notebook's SEQ_LEN: raw_train_split is only
# ~104 weeks after holding out PRED_LEN=39 for honest validation, so
# enc_len+pred_len must stay comfortably under that. 52+39=91 leaves margin
# and still spans a full seasonal cycle (incl. last Christmas). Raise ENC_LEN
# for the FINAL model fit on the full `raw` frame if you want (143 weeks
# available there), but keep it at 52 for anything fit on raw_train_split.
PRED_LEN    = 39       # matches the real Kaggle test horizon exactly
HIDDEN      = 64       # GRN / LSTM hidden size
N_HEADS     = 4        # attention heads
STRIDE      = 1        # data is scarce at enc_len=52 (~13-week margin per series)
BATCH_SIZE  = 128
EPOCHS      = 30       # epochs for the FINAL model (the winner of the sweep below)
LR          = 1e-3
DEVICE      = "cuda" if torch.cuda.is_available() else "cpu"

# calendar covariates are known ahead of time for any forecast horizon,
# so it's safe to feed the FUTURE values of these into the decoder
COV_COLS = ["IsHoliday", "is_superbowl", "is_laborday", "is_thanksgiving",
            "is_christmas_flagged", "is_week_before_christmas",
            "week_sin", "week_cos"]


In [3]:
# ── join raw tables (deep nets skip Group B lags entirely) ─────────────
# NOTE: CalendarFeatures is intentionally NOT applied here. The Pipeline step
# (TFTForecaster) applies it internally on whatever raw frame it's given,
# which is exactly what lets pipeline.predict(raw_test_df) work on an
# untouched test set, per the assignment requirement.
def join_frame(sales_df, stores_df, features_df):
    df = sales_df.merge(stores_df, on="Store", how="left")
    df = df.merge(features_df, on=["Store", "Date"], how="left", suffixes=("", "_feat"))

    # stores_df and features_df both carry Type/Size -> merge creates _x/_y suffixes
    if "Type_x" in df.columns:
        df["Type"] = df["Type_x"]
        df = df.drop(columns=["Type_x"])
    if "Size_x" in df.columns:
        df["Size"] = df["Size_x"]
        df = df.drop(columns=["Size_x"])
    if "Type_y" in df.columns:
        df = df.drop(columns=["Type_y"])
    if "Size_y" in df.columns:
        df = df.drop(columns=["Size_y"])

    if "IsHoliday_feat" in df.columns:
        df["IsHoliday"] = df["IsHoliday_feat"]
        df = df.drop(columns=["IsHoliday_feat"])
    df["Date"] = pd.to_datetime(df["Date"])
    return df

raw = join_frame(train, stores, features)
assert "Type" in raw.columns, "Type column missing after join_frame -- check stores_df merge"
print("raw:", raw.shape, raw["Date"].min(), "→", raw["Date"].max())


raw: (421570, 25) 2010-02-05 00:00:00 → 2012-10-26 00:00:00


In [4]:
# ── time-based split on the RAW frame ───────────────────────────────────
# Splitting `raw` (not a CalendarFeatures-transformed frame) means both
# raw_train_split and raw_val_split stay unprocessed -- exactly the shape
# the Pipeline's .fit()/.predict() are meant to consume, and it lets the
# validation step below double as a rehearsal of the real test.csv call.
raw_train_split, raw_val_split = time_based_split(raw, val_weeks=PRED_LEN)
print("raw_train_split:", raw_train_split.shape, "| raw_val_split:", raw_val_split.shape)


raw_train_split: (305982, 25) | raw_val_split: (115588, 25)


In [5]:
# ── TFT architecture (simplified Temporal Fusion Transformer) ─────────
class GRN(nn.Module):
    """Gated Residual Network -- TFT's basic building block: a small MLP with
    a GLU-style gate and a residual/skip connection, used both to embed the
    static (Store, Dept) context and to enrich the decoder output before
    attention."""
    def __init__(self, in_dim, hidden, out_dim=None, dropout=0.1):
        super().__init__()
        out_dim = out_dim or hidden
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, out_dim)
        self.gate = nn.Linear(out_dim, out_dim * 2)
        self.skip = nn.Linear(in_dim, out_dim) if in_dim != out_dim else nn.Identity()
        self.norm = nn.LayerNorm(out_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        h = torch.relu(self.fc1(x))
        h = self.drop(self.fc2(h))
        a, b = self.gate(h).chunk(2, dim=-1)
        h = a * torch.sigmoid(b)
        return self.norm(h + self.skip(x))


class TFT(nn.Module):
    """Forecaster for the Weekly_Sales channel, weights shared across ALL
    Store x Dept series (one global model, not ~3300 per-series ones) --
    same "one shared net" philosophy as the DLinear notebook's DLinear class.

    Where DLinear uses a moving-average trend/seasonal split + two linear
    seq_len->pred_len maps, TFT instead:
      - embeds (Store, Dept) as static context (GRN-gated),
      - runs an LSTM encoder over the lookback window (target + known covs),
      - runs an LSTM decoder over the FUTURE known covariates (Group A,
        fully known in advance -- the direct analogue of DLinear's cov_head),
      - lets the decoder attend back over the full encoder sequence.
    """
    def __init__(self, n_cov, n_store, n_dept, hidden=64, n_heads=4, dropout=0.1):
        super().__init__()
        self.store_emb = nn.Embedding(n_store, 8)
        self.dept_emb  = nn.Embedding(n_dept, 8)
        self.static_grn = GRN(16, hidden)

        self.enc_in = nn.Linear(n_cov + 1, hidden)   # +1 for past target
        self.dec_in = nn.Linear(n_cov, hidden)
        self.enc_lstm = nn.LSTM(hidden, hidden, batch_first=True)
        self.dec_lstm = nn.LSTM(hidden, hidden, batch_first=True)
        self.static_enrich = GRN(hidden * 2, hidden)
        self.attn = nn.MultiheadAttention(hidden, n_heads, batch_first=True, dropout=dropout)
        self.post_attn_grn = GRN(hidden, hidden)
        self.out = nn.Linear(hidden, 1)

    def forward(self, enc_target, enc_known, dec_known, store_idx, dept_idx):
        # enc_target: [B, enc_len] | enc_known: [B, enc_len, n_cov]
        # dec_known: [B, pred_len, n_cov] | store_idx, dept_idx: [B]
        s = torch.cat([self.store_emb(store_idx), self.dept_emb(dept_idx)], dim=-1)
        s = self.static_grn(s)  # [B, hidden]

        enc_x = self.enc_in(torch.cat([enc_target.unsqueeze(-1), enc_known], dim=-1))
        dec_x = self.dec_in(dec_known)

        enc_out, (h, c) = self.enc_lstm(enc_x)
        dec_out, _ = self.dec_lstm(dec_x, (h, c))

        s_rep = s.unsqueeze(1).expand(-1, dec_out.size(1), -1)
        enriched = self.static_enrich(torch.cat([dec_out, s_rep], dim=-1))

        attn_out, _ = self.attn(enriched, enc_out, enc_out)
        fused = self.post_attn_grn(attn_out + enriched)
        return self.out(fused).squeeze(-1)  # [B, pred_len], normalized scale


In [6]:
# ── training-window dataset + loss (mirrors DLinearTrainDataset) ────────
def to_series_dict(df):
    return {k: g.reset_index(drop=True)
            for k, g in df.sort_values(GROUP_COLS + ["Date"]).groupby(GROUP_COLS)}


class TFTTrainDataset(Dataset):
    """Sliding (enc_len -> pred_len) windows of Weekly_Sales + known
    covariates, pooled across ALL (Store, Dept) series into one dataset so a
    single shared TFT learns globally instead of fitting ~3300 independent
    per-series models -- same pooling strategy as DLinearTrainDataset.

    A window is only emitted if its enc_len+pred_len dates are exactly
    weekly-contiguous -- gappy series are skipped, never bridged. Per-series
    (mu, sigma) z-score stats are fit here and reused at predict time (via
    TFTForecaster.stats_) so scales don't leak across the split. Series whose
    Store/Dept isn't in store_map/dept_map (i.e. wasn't seen when the
    vocabularies were built) are skipped entirely.
    """
    def __init__(self, frame, enc_len, pred_len, cov_cols, store_map, dept_map, stride=1, stats=None):
        self.enc_len, self.pred_len = enc_len, pred_len
        self.stats = {} if stats is None else stats
        fit_stats = stats is None
        self.samples = []
        self.sample_keys = []
        for key, g in to_series_dict(frame).items():
            store, dept = key
            if store not in store_map or dept not in dept_map:
                continue
            sales = g["Weekly_Sales"].to_numpy(dtype=float)
            cov = g[cov_cols].to_numpy(dtype=float)
            dates = g["Date"]
            if fit_stats:
                self.stats[key] = (sales.mean(), sales.std() + 1e-6)
            mu, sigma = self.stats[key]
            L = enc_len + pred_len
            for start in range(0, len(g) - L + 1, stride):
                span = dates.iloc[start:start + L]
                if not (span.diff().dropna() == pd.Timedelta(weeks=1)).all():
                    continue
                enc_target = (sales[start:start + enc_len] - mu) / sigma
                enc_known = cov[start:start + enc_len]
                dec_known = cov[start + enc_len:start + L]
                y = (sales[start + enc_len:start + L] - mu) / sigma
                self.samples.append((enc_target, enc_known, dec_known, y,
                                      store_map[store], dept_map[dept]))
                self.sample_keys.append(key)

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, i):
        enc_target, enc_known, dec_known, y, store_idx, dept_idx = self.samples[i]
        key = self.sample_keys[i]
        mu, sigma = self.stats[key]
        return (torch.tensor(enc_target, dtype=torch.float32),
                torch.tensor(enc_known, dtype=torch.float32),
                torch.tensor(dec_known, dtype=torch.float32),
                torch.tensor(y, dtype=torch.float32),
                torch.tensor(store_idx, dtype=torch.long),
                torch.tensor(dept_idx, dtype=torch.long),
                torch.tensor(mu, dtype=torch.float32),
                torch.tensor(sigma, dtype=torch.float32))


def weighted_l1(pred, target, is_holiday_future):
    """WMAE-shaped training loss: holiday weeks weighted 5x, same weighting
    as the Kaggle metric (src/metric.py:wmae), so the loss the net optimizes
    matches the metric it's judged on. Identical contract to the DLinear
    notebook's weighted_l1 -- must be module-level, not a class method,
    since run_epoch() below calls it directly.
    """
    w = torch.where(is_holiday_future.bool(), 5.0, 1.0)
    return (w * (pred - target).abs()).mean()


In [7]:
# ── standalone training-step function, decoupled from TFTForecaster ─────
# Everything about "how one epoch runs" lives here, so you can change EPOCHS,
# swap optimizers, add a scheduler, or call this directly in a scratch loop
# for fine-tuning -- none of that requires touching TFTForecaster.fit()
# below, which now just calls this in a for-loop (same split as the DLinear
# notebook's run_epoch()).
def run_epoch(model, dl, opt=None, device=None):
    """One pass over `dl`. If `opt` is given, trains (backward + step);
    otherwise runs in eval mode (no grad, no update). Returns
    (avg_normalized_loss, real_scale_wmae).
    """
    device = device or DEVICE
    train_mode = opt is not None
    model.train(train_mode)

    tot_loss, n = 0.0, 0
    all_true, all_pred, all_hol = [], [], []

    for enc_target, enc_known, dec_known, y, store_idx, dept_idx, mu, sigma in dl:
        enc_target = enc_target.to(device)
        enc_known  = enc_known.to(device)
        dec_known  = dec_known.to(device)
        y          = y.to(device)
        store_idx  = store_idx.to(device)
        dept_idx   = dept_idx.to(device)
        mu, sigma  = mu.to(device), sigma.to(device)
        is_hol = dec_known[..., 0]  # IsHoliday must stay COV_COLS[0]

        if train_mode:
            opt.zero_grad()
            pred_norm = model(enc_target, enc_known, dec_known, store_idx, dept_idx)
            loss = weighted_l1(pred_norm, y, is_hol)
            loss.backward()
            opt.step()
        else:
            with torch.no_grad():
                pred_norm = model(enc_target, enc_known, dec_known, store_idx, dept_idx)
                loss = weighted_l1(pred_norm, y, is_hol)

        bs = enc_target.size(0)
        tot_loss += loss.item() * bs
        n += bs

        pred_raw = pred_norm * sigma.unsqueeze(1) + mu.unsqueeze(1)
        true_raw = y * sigma.unsqueeze(1) + mu.unsqueeze(1)
        all_pred.append(pred_raw.detach().cpu().numpy().ravel())
        all_true.append(true_raw.detach().cpu().numpy().ravel())
        all_hol.append(is_hol.detach().cpu().numpy().ravel())

    y_true = np.concatenate(all_true)
    y_pred = np.concatenate(all_pred)
    is_hol_all = np.concatenate(all_hol)
    return tot_loss / n, wmae(y_true, y_pred, is_hol_all)


In [8]:
# ── wrap TFT as a single sklearn-compatible Pipeline step ───────────
from sklearn.base import BaseEstimator, RegressorMixin
from sklearn.pipeline import Pipeline

class TFTForecaster(BaseEstimator, RegressorMixin):
    """Makes the trained TFT runnable end-to-end on a RAW, unprocessed frame
    (same raw shape as join_frame()'s output: Date, Store, Dept, IsHoliday,
    MarkDown1-5 w/ NaNs, Type, Size, macro cols -- no CalendarFeatures
    applied, no Weekly_Sales normalization done). Mirrors DLinearForecaster.

    fit(X_raw, y): applies CalendarFeatures internally, builds the
    Store/Dept embedding vocabularies from whatever is seen during this
    fit() call, trains the TFT net, and snapshots the per-series
    (Store, Dept) history + (mu, sigma) stats needed to build lookback
    windows at predict time -- mirrors how LagFeatures.fit() snapshots
    history in features.py.

    predict(X_raw): applies CalendarFeatures internally, pulls each row's
    enc_len lookback from the stored TRAIN history (never from y, since
    predict never receives y), builds the decoder's known-future covariates
    from the row's own (already-known) calendar features, runs the net,
    de-normalizes, and returns predictions aligned to X_raw's row order.
    Rows whose series has insufficient history, or whose Store/Dept was
    never seen during fit, return NaN rather than raising, so predict()
    never crashes on unseen (Store, Dept) combos.
    """
    def __init__(self, enc_len=52, pred_len=39, cov_cols=None,
                 hidden=64, n_heads=4, epochs=30, lr=1e-3, batch_size=128,
                 stride=1, device=None, verbose=False):
        self.enc_len, self.pred_len = enc_len, pred_len
        self.cov_cols = cov_cols or COV_COLS
        self.hidden, self.n_heads = hidden, n_heads
        self.epochs, self.lr = epochs, lr
        self.batch_size, self.stride = batch_size, stride
        self.device = device or ("cuda" if torch.cuda.is_available() else "cpu")
        self.verbose = verbose

    def _featurize(self, X_raw):
        return CalendarFeatures().fit_transform(X_raw).sort_values(
            GROUP_COLS + ["Date"]).reset_index(drop=True)

    def fit(self, X_raw, y=None):
        # attach y to X_raw BEFORE featurizing -- _featurize() sorts by
        # (Store, Dept, Date) and resets the index, so y must be glued to
        # its row before that reorder (same fix as DLinearForecaster.fit()).
        X_in = X_raw.copy()
        if y is not None:
            X_in["Weekly_Sales"] = np.asarray(y)
        df = self._featurize(X_in)

        # Store/Dept vocabularies are built from whatever this fit() call
        # sees -- analogous to how DLinearTrainDataset snapshots (mu, sigma)
        # per series inside fit(), not at __init__ time.
        store_ids = sorted(df["Store"].unique())
        dept_ids  = sorted(df["Dept"].unique())
        self.store_map_ = {s: i for i, s in enumerate(store_ids)}
        self.dept_map_  = {d: i for i, d in enumerate(dept_ids)}

        ds = TFTTrainDataset(df, self.enc_len, self.pred_len, self.cov_cols,
                              self.store_map_, self.dept_map_, stride=self.stride)
        if len(ds) == 0:
            n_weeks = df.groupby(GROUP_COLS)["Date"].nunique().max() if len(df) else 0
            raise ValueError(
                f"TFTTrainDataset produced 0 windows. enc_len+pred_len="
                f"{self.enc_len + self.pred_len} weeks are needed per window, but the "
                f"longest series in this fit() call only has {n_weeks} contiguous weeks. "
                f"Lower enc_len (or pred_len) so enc_len+pred_len <= the weeks available "
                f"in whatever frame you're calling .fit() on."
            )
        loader = DataLoader(ds, batch_size=self.batch_size, shuffle=True, drop_last=True)

        self.model_ = TFT(len(self.cov_cols), len(self.store_map_), len(self.dept_map_),
                           hidden=self.hidden, n_heads=self.n_heads).to(self.device)
        opt = torch.optim.Adam(self.model_.parameters(), lr=self.lr)

        # the epoch loop just calls the standalone run_epoch() -- change
        # self.epochs, swap opt, add a scheduler, whatever, without touching
        # this method at all. epoch_history_ lets a caller (e.g. the sweep
        # cell) log per-epoch metrics to mlflow.
        self.epoch_history_ = []
        for epoch in range(self.epochs):
            tr_loss, tr_wmae = run_epoch(self.model_, loader, opt, device=self.device)
            self.epoch_history_.append({"epoch": epoch, "train_loss": tr_loss, "train_wmae": tr_wmae})
            if self.verbose:
                print(f"  epoch {epoch:>3d}  train_loss={tr_loss:.4f}  train_wmae={tr_wmae:.2f}")

        # snapshot train history + per-series stats -- same role as
        # LagFeatures.history_, needed because predict() gets no y
        self.history_ = {k: g.reset_index(drop=True)
                          for k, g in df.groupby(GROUP_COLS)}
        self.stats_ = ds.stats
        # fallback for rows predict() can't cover (series with no usable history
        # window). Without this the pipeline emits NaN, which would land straight
        # in submission.csv and be rejected by Kaggle.
        self.series_mean_ = df.groupby(GROUP_COLS)["Weekly_Sales"].mean().to_dict()
        self.global_mean_ = float(df["Weekly_Sales"].mean())
        # every covariate is a pure function of Date, identical across series, so
        # this table lets predict() rebuild an exact encoder window even for weeks
        # a given series never recorded
        self.cov_by_date_ = (df.drop_duplicates("Date")
                               .set_index("Date")[self.cov_cols].sort_index())
        return self

    def predict(self, X_raw):
        # _featurize() sorts by (Store, Dept, Date), so remember where every row
        # started out and put the predictions back in the caller's row order at
        # the end. Otherwise this only works by luck on an already-sorted frame.
        feats = CalendarFeatures().fit_transform(X_raw)
        feats["_pos"] = np.arange(len(feats))
        df = feats.sort_values(GROUP_COLS + ["Date"]).reset_index(drop=True)

        # every covariate is a pure function of Date (one distinct value per date
        # across all stores), so a single date -> covariates table covers both the
        # history weeks and the forecast weeks, including weeks a given series
        # never recorded
        cov_table = pd.concat([self.cov_by_date_,
                                df.drop_duplicates("Date").set_index("Date")[self.cov_cols]])
        cov_table = cov_table[~cov_table.index.duplicated()].sort_index()

        self.model_.eval()
        preds = np.full(len(df), np.nan, dtype=float)
        min_real = max(4, self.enc_len // 2)

        for key, g in df.groupby(GROUP_COLS):
            store, dept = key
            hist = self.history_.get(key)
            if (hist is None or key not in self.stats_
                    or store not in self.store_map_ or dept not in self.dept_map_):
                continue  # unseen series -> mean fallback below
            mu, sigma = self.stats_[key]
            g_sorted = g.sort_values("Date")

            # the net was trained to forecast the pred_len weeks immediately after a
            # contiguous seq_len-week window, so rebuild exactly that: the seq_len
            # calendar weeks ending the week before this series' horizon starts.
            origin = g_sorted["Date"].iloc[0] - pd.Timedelta(weeks=1)
            grid = pd.date_range(end=origin, periods=self.enc_len, freq="W-FRI")

            # weeks the series never recorded get interpolated. Taking the last
            # seq_len OBSERVATIONS instead (tail) would hand a gappy window to a net
            # that only ever saw consecutive weeks, silently misdating every input.
            past = hist[hist["Date"] <= origin]
            sales = past.set_index("Date")["Weekly_Sales"].reindex(grid)
            if sales.notna().sum() < min_real:
                continue  # too little real history to trust -> mean fallback
            sales = sales.interpolate(limit_direction="both").to_numpy(dtype=float)

            # map each row to its forecast step by DATE. pred_norm[i] is the forecast
            # for week i+1 after `origin`; indexing it positionally by row would read
            # the wrong step for any series with a missing week.
            step = ((g_sorted["Date"] - origin).dt.days // 7 - 1).to_numpy()
            ok = (step >= 0) & (step < self.pred_len)
            if not ok.any():
                continue

            dec_dates = pd.date_range(start=origin + pd.Timedelta(weeks=1),
                                       periods=self.pred_len, freq="W-FRI")
            cov_future = cov_table.reindex(dec_dates).ffill().bfill().to_numpy(dtype=float)

            enc_cov = cov_table.reindex(grid).ffill().bfill().to_numpy(dtype=float)
            enc_target = torch.tensor((sales - mu) / sigma,
                                       dtype=torch.float32, device=self.device).view(1, self.enc_len)
            enc_known = torch.tensor(enc_cov, dtype=torch.float32,
                                      device=self.device).view(1, self.enc_len, -1)
            dec_window = torch.tensor(cov_future, dtype=torch.float32, device=self.device)
            store_idx = torch.tensor([self.store_map_[store]], dtype=torch.long, device=self.device)
            dept_idx  = torch.tensor([self.dept_map_[dept]], dtype=torch.long, device=self.device)
            with torch.no_grad():
                pred_norm = np.atleast_1d(
                    self.model_(enc_target, enc_known, dec_window.unsqueeze(0),
                                 store_idx, dept_idx).squeeze(0).cpu().numpy())
            preds[df.index.get_indexer(g_sorted.index[ok])] = pred_norm[step[ok]] * sigma + mu

        # fraction the net itself covered, before the mean fallback fills the rest
        nan = np.isnan(preds)
        self.last_coverage_ = float(1.0 - nan.mean())
        if nan.any():
            keys = list(zip(df["Store"].to_numpy(), df["Dept"].to_numpy()))
            preds[nan] = [self.series_mean_.get(keys[i], self.global_mean_)
                          for i in np.flatnonzero(nan)]

        out = np.empty(len(df), dtype=float)
        out[df["_pos"].to_numpy()] = preds
        return out


def make_tft_pipeline(enc_len, pred_len=PRED_LEN, hidden=HIDDEN, n_heads=N_HEADS,
                       lr=LR, epochs=EPOCHS, batch_size=BATCH_SIZE, stride=STRIDE):
    """Factory so the sweep cell below can build a fresh Pipeline per config
    without repeating the TFTForecaster(...) call each time."""
    return Pipeline([
        ("tft", TFTForecaster(enc_len=enc_len, pred_len=pred_len,
                               cov_cols=COV_COLS, hidden=hidden, n_heads=n_heads,
                               epochs=epochs, lr=lr, batch_size=batch_size,
                               stride=stride)),
    ])


In [9]:
# ── hyperparameter sweep: try a few configs, log each as a lightweight
# mlflow run (metrics only -- no point pickling every candidate net), then
# pick the winner by val_wmae. Only the winner gets the full Pipeline
# artifact logged, in the next cells. ───────────────────────────────────
max_train_weeks = raw_train_split.groupby(GROUP_COLS)["Date"].nunique().max()
print(f"longest series in raw_train_split: {max_train_weeks} weeks "
      f"(enc_len+pred_len must stay <= this)")

SWEEP_EPOCHS = 15   # shorter than the final EPOCHS -- just enough to rank configs

SEARCH_SPACE = [
    dict(enc_len=39, hidden=64, n_heads=4, lr=1e-3),
    dict(enc_len=52, hidden=64, n_heads=4, lr=1e-3),
    dict(enc_len=52, hidden=96, n_heads=4, lr=5e-4),
    dict(enc_len=65, hidden=64, n_heads=2, lr=1e-3),
]

mlflow.set_experiment("TemporalFusionTransformer_Training")
raw_val_features = raw_val_split.drop(columns=["Weekly_Sales"])

sweep_results = []
for cfg in SEARCH_SPACE:
    if cfg["enc_len"] + PRED_LEN > max_train_weeks:
        print(f"skip {cfg}: enc_len+pred_len={cfg['enc_len']+PRED_LEN} > {max_train_weeks} weeks available")
        continue

    pipe = make_tft_pipeline(enc_len=cfg["enc_len"], hidden=cfg["hidden"],
                              n_heads=cfg["n_heads"], lr=cfg["lr"], epochs=SWEEP_EPOCHS)
    pipe.fit(raw_train_split, raw_train_split["Weekly_Sales"])

    val_preds = pipe.predict(raw_val_features)
    assert np.isfinite(val_preds).all(), "predict() returned NaN/inf"
    # score every row, not just the ones the net covered -- masking the uncovered
    # rows out would quietly drop the hardest ~2% of the split from the metric
    coverage = pipe.named_steps["tft"].last_coverage_
    val_wmae = wmae(raw_val_split["Weekly_Sales"], val_preds, raw_val_split["IsHoliday"])

    run_name = f"tft_sweep_enc{cfg['enc_len']}_h{cfg['hidden']}_heads{cfg['n_heads']}_lr{cfg['lr']}"
    with mlflow.start_run(run_name=run_name):
        mlflow.log_params(dict(**cfg, pred_len=PRED_LEN, stride=STRIDE,
                                batch_size=BATCH_SIZE, epochs=SWEEP_EPOCHS,
                                feature_group="A_only_calendar", sweep=True))
        # per-epoch curve, available because TFTForecaster.fit() stashes it
        # via run_epoch() -- lets you see convergence per config in the
        # MLflow UI, not just the final number
        for h in pipe.named_steps["tft"].epoch_history_:
            mlflow.log_metrics({"train_loss": h["train_loss"], "train_wmae": h["train_wmae"]}, step=h["epoch"])
        mlflow.log_metric("val_wmae", val_wmae)
        mlflow.log_metric("val_coverage", coverage)
        run_id = mlflow.active_run().info.run_id

    print(f"{cfg}  epochs={SWEEP_EPOCHS} -> val_wmae={val_wmae:.2f}  coverage={coverage:.1%}  run={run_id}")
    sweep_results.append(dict(cfg=cfg, val_wmae=val_wmae, coverage=coverage, run_id=run_id))
    del pipe  # free the net + history_ copy before the next config trains

sweep_results.sort(key=lambda r: r["val_wmae"])
if not sweep_results:
    raise ValueError(
        "Every config in SEARCH_SPACE was skipped (enc_len+pred_len exceeded "
        f"max_train_weeks={max_train_weeks}). Lower the enc_len values in SEARCH_SPACE."
    )
print("\nleaderboard (best first):")
for r in sweep_results:
    print(f"  val_wmae={r['val_wmae']:.2f}  coverage={r['coverage']:.1%}  {r['cfg']}")

best = sweep_results[0]
print("\nBEST CONFIG:", best["cfg"], "| sweep val_wmae:", round(best["val_wmae"], 2))


longest series in raw_train_split: 104 weeks (enc_len+pred_len must stay <= this)
🏃 View run tft_sweep_enc39_h64_heads4_lr0.001 at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/8/runs/31dcea79c8184041be6c9a70c72980c7
🧪 View experiment at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/8
{'enc_len': 39, 'hidden': 64, 'n_heads': 4, 'lr': 0.001}  epochs=15 -> val_wmae=2051.19  coverage=98.0%  run=31dcea79c8184041be6c9a70c72980c7
🏃 View run tft_sweep_enc52_h64_heads4_lr0.001 at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/8/runs/8400740411d445a88b3fca32d375a991
🧪 View experiment at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/8
{'enc_len': 52, 'hidden': 64, 'n_heads': 4, 'lr': 0.001}  epochs=15 -> val_wmae=2162.44  coverage=97.9%  run=8400740411d445a88b3fca32d375a991
🏃 View run tft_sweep_enc52_h96_heads4_lr0.0005 at: https://dagshub.com/llikl23/walmart-sales-forecasting.m

In [10]:
# ── refit the winning config at full EPOCHS -- the short SWEEP_EPOCHS run
# above was only for ranking, the model you actually log should be trained
# properly. This becomes `tft_pipeline`, same name the next cell expects.
tft_pipeline = make_tft_pipeline(enc_len=best["cfg"]["enc_len"], hidden=best["cfg"]["hidden"],
                                  n_heads=best["cfg"]["n_heads"], lr=best["cfg"]["lr"], epochs=EPOCHS)
tft_pipeline.fit(raw_train_split, raw_train_split["Weekly_Sales"])
print("refit done with best config:", best["cfg"], "| epochs:", EPOCHS)


refit done with best config: {'enc_len': 39, 'hidden': 64, 'n_heads': 4, 'lr': 0.001} | epochs: 30


In [11]:
# ── validate on the RAW val split, then refit on FULL train and log THAT ──
# `tft_pipeline` was fitted on raw_train_split only, so it gives an honest,
# leakage-free validation number. But its internal history_ stops at the val
# cutoff, 40 weeks before test.csv begins, so it physically cannot forecast the
# real test block. The artifact we log has to be refit on the full train frame,
# exactly like the xgboost notebook does.
raw_val_features = raw_val_split.drop(columns=["Weekly_Sales"])
val_preds = tft_pipeline.predict(raw_val_features)

assert np.isfinite(val_preds).all(), "predict() returned NaN/inf on the val split"
coverage = tft_pipeline.named_steps["tft"].last_coverage_
val_wmae = wmae(raw_val_split["Weekly_Sales"], val_preds, raw_val_split["IsHoliday"])
print(f"val coverage: {coverage:.1%}  |  val WMAE: {val_wmae:.2f}")
if coverage < 0.9:
    print("WARNING: low coverage -- many (Store, Dept) series don't have "
          "enough recorded weeks before the val cutoff; consider a smaller enc_len.")

# refit the winning config on the whole train frame
final_pipeline = make_tft_pipeline(enc_len=best["cfg"]["enc_len"], hidden=best["cfg"]["hidden"],
                                    n_heads=best["cfg"]["n_heads"], lr=best["cfg"]["lr"],
                                    epochs=EPOCHS)
final_pipeline.fit(raw, raw["Weekly_Sales"])

# the real thing: raw, unprocessed test frame straight through the Pipeline
raw_test = join_frame(test, stores, features)
test_preds = final_pipeline.predict(raw_test)
assert np.isfinite(test_preds).all(), "final pipeline returned NaN/inf on test"
test_coverage = final_pipeline.named_steps["tft"].last_coverage_
print(f"test coverage: {test_coverage:.1%} | sample test preds: {test_preds[:3].round(1)}")

# move the net to CPU before pickling so the logged model loads on any
# machine (Kaggle scoring / a teammate's laptop may not have a GPU)
final_pipeline.named_steps["tft"].model_.to("cpu")
final_pipeline.named_steps["tft"].device = "cpu"

# log the winning config's ACTUAL hyperparameters via get_params(), not the
# config-cell globals -- those still hold the sweep's default/first values
# and would silently mismatch whatever the sweep actually picked.
fitted_params = final_pipeline.named_steps["tft"].get_params()

mlflow.set_experiment("TemporalFusionTransformer_Training")
with mlflow.start_run(run_name="tft_best"):
    mlflow.log_params({**fitted_params, "feature_group": "A_only_calendar",
                        "cov_cols": ",".join(COV_COLS), "refit_on": "full_train",
                        "selected_via_sweep_run_id": best["run_id"]})
    for h in final_pipeline.named_steps["tft"].epoch_history_:
        mlflow.log_metrics({"train_loss": h["train_loss"], "train_wmae": h["train_wmae"]}, step=h["epoch"])
    mlflow.log_metric("val_wmae", val_wmae)        # measured on the train_split fit
    mlflow.log_metric("val_coverage", coverage)
    mlflow.log_metric("test_coverage", test_coverage)
    model_info = mlflow.sklearn.log_model(final_pipeline, artifact_path="model", serialization_format="cloudpickle")
    run_id = mlflow.active_run().info.run_id
    print("logged pipeline run:", run_id, "| val WMAE:", val_wmae, "| config:", best["cfg"])


val coverage: 98.0%  |  val WMAE: 2010.47
test coverage: 98.8% | sample test preds: [28678.4 23595.9 20336.2]


2026/07/10 11:49:07 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/07/10 11:49:15 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html
2026/07/10 11:49:28 WARNING mlflow.utils.requirements_utils: Found torch version (2.11.0+cu128) contains a local version label (+cu128). MLflow logged a pip requirement for this package as 'torch==2.11.0' without the local version label to make it installable from PyPI. To specify pip requirements containing local version labels, please use `conda_env` or `pip_requirements`.


logged pipeline run: 574709d190974e6bbf31f4e1bdbe0c0a | val WMAE: 2010.4740947207551 | config: {'enc_len': 39, 'hidden': 64, 'n_heads': 4, 'lr': 0.001}
🏃 View run tft_best at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/8/runs/574709d190974e6bbf31f4e1bdbe0c0a
🧪 View experiment at: https://dagshub.com/llikl23/walmart-sales-forecasting.mlflow/#/experiments/8


In [12]:
# sanity check: reload exactly the way a grader / submission script would,
# and confirm it runs on a fully raw, unprocessed frame with zero manual prep.
# Uses model_info.model_uri (what log_model() actually returned) instead of
# hand-building "runs:/{run_id}/model" -- recent MLflow versions track logged
# models as their own entity, so the reconstructed runs:/ path can fail to
# resolve even when logging itself succeeded.
try:
    loaded = mlflow.sklearn.load_model(model_info.model_uri)
    sanity_preds = loaded.predict(raw_test)
    print("reload sanity check, max abs diff vs original test preds:",
          float(np.max(np.abs(sanity_preds - test_preds))))
except Exception as e:
    print(f"WARNING: model logged successfully, but the in-notebook reload "
          f"check failed ({type(e).__name__}: {e}). This doesn't mean the "
          f"artifact is broken -- try loading it from the MLflow UI/registry "
          f"directly to confirm before assuming the run is bad.")


reload sanity check, max abs diff vs original test preds: 9.189073092784383
